## working with textract -> llm workflow

In [5]:
import os
import base64
import sys
from dotenv import load_dotenv
from typing import Any, Dict, List, Optional, Tuple
from PIL import Image   # noqa: F401
from pydantic import BaseModel
from pydantic import Field
from pydantic_settings import BaseSettings
import oracledb
import io
import cv2
import fitz
import numpy as np
import boto3
from botocore.config import Config
import json
import re
import time
from tqdm import tqdm
import mariadb
import pandas as pd
from collections import defaultdict
from dataclasses import dataclass, field
#strands agetinc workflow

#dont limit the visualization of all the colluns of a pandas dataframe
pd.set_option("display.max_columns", None)

# Add project root to Python path so we can import from app module
project_root = os.path.dirname(os.getcwd())
if project_root not in sys.path:
    sys.path.append(project_root)
# Now we can import from app (after adding to sys.path)
from app.utils.logger import get_logger
# Load environment variables from the project root directory
env_path = os.path.join(project_root, '.env')
load_dotenv(env_path)

logger = get_logger(name=__name__)

## credentials config

In [6]:
@dataclass
class AppConstants:
    BEDROCK_DEFAULT_MODEL_ID: str = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"
    BEDROCK_DEFAULT_MODEL_VERSION: str = "bedrock-2023-05-31"
    DEFAULT_PROMPTS_DIR: str = "prompts/"
    S3_BUCKET_NAME: str = "agente-ai-carteirinha"
    S3_RESULTS_PREFIX: str = "resultados"
    S3_DEBUG_PREFIX: str = "debug"
    STREAMING: bool = False
    CACHE_PROMPT = "default"
    RETRIES: Dict[str, int] = field(default_factory=lambda: {"max_attempts": 3, "mode": "standard"})
    CONNECTION_TIMEOUT: int = 5
    READ_TIMEOUT: int = 60
    TEMPERATURE: float = 1
    TOP_P: float = 0.95
    MAX_TOKENS: int = 4096
    BUDGET_TOKENS: int = 2048

In [7]:
class Settings(BaseSettings):
    """Carrega e valida as configurações a partir de variáveis de ambiente."""

    ORACLE_USER: str
    ORACLE_PASSWORD: str
    ORACLE_DSN: str
    ORACLE_INSTANT_CLIENT_PATH: Optional[str] = Field(
        None, alias="oracle_instant_client_path"
    )
    AWS_ACCESS_KEY_ID: str
    AWS_SECRET_ACCESS_KEY: str
    AWS_BEDROCK_REGION: str
    BEDROCK_MODEL_ID: str = AppConstants.BEDROCK_DEFAULT_MODEL_ID
    BEDROCK_MODEL_VERSION: str = AppConstants.BEDROCK_DEFAULT_MODEL_VERSION
    AWS_SERVICE_NAME: str
    MARIADB_USER: str
    MARIADB_PASSWORD: str
    MARIADB_HOST: str
    MARIADB_PORT: int = 3306
    MARIADB_DATABASE: str
    API_BASE_URL: Optional[str] = Field(None, alias="api_base_url")
    API_USERNAME: Optional[str] = Field(None, alias="username")
    API_PASSWORD: Optional[str] = Field(None, alias="password")

    class Config:
        env_file = ".env"
        env_file_encoding = "utf-8"

In [ ]:
def criar_boto3_client(
    service_name: str, settings: Settings, config: Optional[Config] = None
) -> boto3.client:
    try:
        logger.info(
            f"Criando cliente {service_name.upper()} para a região: {settings.AWS_BEDROCK_REGION}..."
        )
        client = boto3.client(
            service_name,
            region_name=settings.AWS_BEDROCK_REGION,
            aws_access_key_id=settings.AWS_ACCESS_KEY_ID,
            aws_secret_access_key=settings.AWS_SECRET_ACCESS_KEY,
            config=config,
        )
        logger.info(f"Cliente {service_name.upper()} criado com sucesso.")
        return client
    except Exception as e:
        logger.critical(f"Não foi possível criar o cliente {service_name.upper()}: {e}")
        raise


## geting the service ready to use
### model configuration

In [ ]:
easy_prompt = "what are llm models?"

In [ ]:
def safe_extract_json(raw_text: str) -> dict:
    match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', raw_text, re.DOTALL) \
        or re.search(r'\{.*?\}', raw_text, re.DOTALL)
    
    if not match:
        raise ValueError("Nenhum JSON encontrado na resposta do modelo.")

    json_str = match.group(1) if match.lastindex else match.group(0)
    json_str = json_str.strip()

    try:
        return json.loads(json_str)
    except json.JSONDecodeError as e:
        logger.warning(f"JSON inválido detectado: {e}")
        # tenta corrigir
        cleaned = re.sub(r'^[^\{]*', '', json_str)
        cleaned = re.sub(r'[^\}]*$', '', cleaned)
        return json.loads(cleaned)

In [ ]:
system_prompt = fr"""
Você é um especialista em extração estruturada de dados.  
Sua tarefa: analisar o texto e retornar **apenas** as informações de carteirinhas de convênio de saúde em JSON válido.
⚠️ Retorne **somente JSON**, sem explicações, comentários ou texto adicional.

Campos a extrair:
- "convenio": Nome do convênio
- "plano": Tipo do plano (ex.: "Plano Prata")
- "nome_pessoa": Nome completo do beneficiário
- "numero_carteirinha": Número da carteirinha processado conforme regras abaixo

Regras de limpeza:
1. Remover caracteres especiais usando regex `[^a-zA-Z0-9\s]`
2. Remover espaços no início e fim
3. Aplicar regras especiais por convênio antes de validar tamanho
4. Validar tamanho conforme lista de mapeamento
5. Se algum campo não puder ser identificado ou validado, retornar null

Regras gerais:
- Se vier nomes de duas pessoas, considere que o nome a ser definido nao é o nome relativo ao titular.

Regras especiais por convênio:
- CEMIG SAUDE: Se houver dois números, use a matrícula do beneficiário (não a matrícula antiga)
- SUL AMERICA: Se o número tiver mais de 17 dígitos, remover os 3 primeiros dígitos e manter os 17 últimos.
- Outros convênios: Validar tamanho conforme tabela; se não estiver na lista, retornar null

Tabela de mapeamento (convênio, número de dígitos esperado):
[
("STELLANTIS SAUDE MG", 17),
("SUL AMERICA", "variavel"),
("CASSI", 16),
("CAIXA ECONOMICA FEDERAL", 11),
("BLUE COMPANY", 16),
("POSTAL SAUDE - CORREIOS", 16),
("IPSM", 16),
("UNIMED SEGUROS", 16),
("BRADESCO", 15),
("BRADESCO OPERADORA", 15),
("PLAN ASSISTE - MPF", 14),
("CARE PLUS", 12),
("PETROBRAS - REGAP", 12),
("VALE - AMS", 12),
("FUNDAFFEMG", 12),
("CEMIG SAUDE", "variavel"),
("VALE - PASA", 10),
("AMIL", 9),
("AMIL VM (ANTIGA GOLDEN CROSS)", 9),
("COPASS", 8),
("SPA SAUDE", 5)
]

Exemplo:
Texto: "Paciente João da Silva, convênio SUL AMERICA, carteirinha 12345678901234567890"
JSON esperado:
{{
  "convenio": "SUL AMERICA",
  "plano": null,
  "nome_pessoa": "João da Silva",
  "numero_carteirinha": "45678901234567890"
}}

"""

In [ ]:
import json
with open("system_prompt.json", "w", encoding="utf-8") as f:
    json.dump({"system_prompt": system_prompt}, f, ensure_ascii=False, indent=2)

In [ ]:
system_prompt_2 = fr"""
Você é um especialista em extração estruturada de dados de carteirinhas de convênio de saúde.  
Sua tarefa: analisar o texto e retornar **apenas** um JSON válido contendo as informações do **beneficiário da carteirinha**.  
⚠️ Retorne **somente JSON**, sem explicações, comentários ou texto adicional.

Campos a extrair:
- "convenio": Nome do convênio
- "plano": Tipo do plano (ex.: "Plano Prata"), ou null se não identificado
- "nome_pessoa": Nome completo do **beneficiário** (não do titular ou de outra pessoa)
- "numero_carteirinha": Número da carteirinha processado conforme regras abaixo

Regras de extração de nome:
1. Sempre escolher o **nome do beneficiário**, mesmo que outro nome apareça próximo de “titular”.
2. Se houver múltiplos nomes, ignorar nomes que aparecem associados a “titular”, “titularidade” ou outros familiares.
3. Remover caracteres especiais e espaços extras.

Regras de limpeza do número da carteirinha:
1. Remover caracteres especiais usando regex `[^a-zA-Z0-9\s]`
2. Remover espaços no início e fim
3. Aplicar regras especiais por convênio antes de validar tamanho
4. Validar tamanho conforme lista de mapeamento
5. Se algum campo não puder ser identificado ou validado, retornar null

Regras especiais por convênio:
- CEMIG SAUDE: Se houver dois números, use a matrícula do beneficiário (não a matrícula antiga)
- SUL AMERICA: Se o número tiver mais de 17 dígitos, remover os 3 primeiros dígitos e manter os 17 últimos.
- Outros convênios: Validar tamanho conforme tabela; se não estiver na lista, retornar null

Tabela de mapeamento (convênio, número de dígitos esperado):
[
("STELLANTIS SAUDE MG", 17),
("SUL AMERICA", "variavel"),
("CASSI", 16),
("CAIXA ECONOMICA FEDERAL", 11),
("BLUE COMPANY", 16),
("POSTAL SAUDE - CORREIOS", 16),
("IPSM", 16),
("UNIMED SEGUROS", 16),
("BRADESCO", 15),
("BRADESCO OPERADORA", 15),
("PLAN ASSISTE - MPF", 14),
("CARE PLUS", 12),
("PETROBRAS - REGAP", 12),
("VALE - AMS", 12),
("FUNDAFFEMG", 12),
("CEMIG SAUDE", "variavel"),
("VALE - PASA", 10),
("AMIL", 9),
("AMIL VM (ANTIGA GOLDEN CROSS)", 9),
("COPASS", 8),
("SPA SAUDE", 5)
]

Exemplo:
Texto: "Paciente João da Silva, convênio SUL AMERICA, carteirinha 12345678901234567890"
JSON esperado:
{{
  "convenio": "SUL AMERICA",
  "plano": null,
  "nome_pessoa": "João da Silva",
  "numero_carteirinha": "45678901234567890"
}}
"""


In [ ]:
# api format

from collections import defaultdict

class TextractKVExtractor:
    def __init__(self, textract_client):
        """
        textract_client: boto3 Textract client (already configured)
        """
        self.client = textract_client

    def _get_kv_map(self, file_bytes):
        response = self.client.analyze_document(
            Document={'Bytes': file_bytes},
            FeatureTypes=['FORMS']
        )
        blocks = response['Blocks']
        key_map, value_map, block_map = {}, {}, {}

        for block in blocks:
            block_map[block['Id']] = block
            if block['BlockType'] == 'KEY_VALUE_SET':
                if 'KEY' in block['EntityTypes']:
                    key_map[block['Id']] = block
                else:
                    value_map[block['Id']] = block
        return key_map, value_map, block_map

    def _find_value_block(self, key_block, value_map):
        for rel in key_block.get('Relationships', []):
            if rel['Type'] == 'VALUE':
                for value_id in rel['Ids']:
                    return value_map.get(value_id)
        return None

    def _get_text(self, block, block_map):
        if not block:
            return ''
        text = ''
        for rel in block.get('Relationships', []):
            if rel['Type'] == 'CHILD':
                for child_id in rel['Ids']:
                    child = block_map.get(child_id, {})
                    if child.get('BlockType') == 'WORD':
                        text += child.get('Text', '') + ' '
                    if child.get('BlockType') == 'SELECTION_ELEMENT' and child.get('SelectionStatus') == 'SELECTED':
                        text += 'X '
        return text.strip()

    def _get_kv_relationship(self, key_map, value_map, block_map):
        kvs = defaultdict(list)
        for key_id, key_block in key_map.items():
            value_block = self._find_value_block(key_block, value_map)
            key_text = self._get_text(key_block, block_map)
            value_text = self._get_text(value_block, block_map)
            kvs[key_text].append(value_text)
        return kvs
    
    def _extract_all_text(self, file_bytes: bytes) -> str:
        """
        Extracts all text from the document, including text not associated with keys.
        """
        response = self.client.detect_document_text(Document={'Bytes': file_bytes})
        text_blocks = [
            block['Text']
            for block in response['Blocks']
            if block['BlockType'] == 'LINE'
        ]
        return "\n".join(text_blocks)

    def run(self, file_path=None, file_bytes=None, extract_full_text=False):
        """
        Run the extraction.
        Provide either `file_path` or `file_bytes`.
        If extract_full_text=True, returns all text (not just key-value pairs).
        """
        if file_path:
            with open(file_path, 'rb') as f:
                file_bytes = f.read()
        if not file_bytes:
            raise ValueError("You must provide file_path or file_bytes.")

        if extract_full_text:
            return self._extract_all_text(file_bytes)

        key_map, value_map, block_map = self._get_kv_map(file_bytes)
        kvs = self._get_kv_relationship(key_map, value_map, block_map)
        return str(kvs)


### prepare the input pdf
The goal is to get a list of 1 or more png or jpeg bytes

In [ ]:
# image_utils
MAX_IMAGES_PER_BLOB = 20  # Maximum number of images per blob
def converter_blob_para_imagens(blob: bytes, extensao: str) -> List[Image.Image]:
    imagens = []
    ext = extensao.lower().strip(".") if extensao else ""
    try:
        if ext == "pdf":
            with fitz.open(stream=blob, filetype="pdf") as pdf_doc:
                logger.info(f"Processando PDF com {len(pdf_doc)} página(s)...")
                for pagina in pdf_doc: # limit to 20 pages
                    if len(imagens) >= MAX_IMAGES_PER_BLOB:
                        logger.warning(f"⚠️ BLOB has reached the maximum limit of {MAX_IMAGES_PER_BLOB} images.")
                        break
                    pix = pagina.get_pixmap(matrix=fitz.Matrix(3.0, 3.0), alpha=False)
                    imagens.append(Image.open(io.BytesIO(pix.tobytes("png"))))
        elif ext in ["jpg", "jpeg", "png", "bmp"]:
            imagens.append(Image.open(io.BytesIO(blob)))
        else:
            logger.warning(f"Formato de arquivo não suportado: '{ext}'.")
    except Exception as e:
        logger.error(f"Erro ao converter BLOB para imagem (ext: .{ext}): {e}")
    return imagens


def aplicar_clahe(imagem: Image.Image) -> Image.Image:
    try:
        imagem_cv = cv2.cvtColor(np.array(imagem), cv2.COLOR_RGB2BGR)
        imagem_cinza = cv2.cvtColor(imagem_cv, cv2.COLOR_BGR2GRAY)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        return Image.fromarray(clahe.apply(imagem_cinza))
    except Exception:
        return imagem

def imagens_para_bytes(images: Image.Image, formato: str = "PNG") -> List[bytes]:
    buffer = io.BytesIO()
    images.save(buffer, format=formato)
    return [buffer.getvalue()]


In [ ]:


from collections import defaultdict



def process_blobs(blobs_and_id: Dict[str, bytes], img_enhancement: bool = True) -> Dict[str, list[bytes]]:
    """get the pdf blob, in bytes, convert to png images, apply enchacement if wanted, and get the png bytes.
    args: blobs_and_id : Dict[str, list[bytes]]: Dictionary mapping IDs to their corresponding byte content.
          img_enhancement (bool): Flag indicating whether to apply image enhancement.
    output: defaultdict(any, list): A dictionary mapping IDs - str to their corresponding PNG image bytes.
    """
    byte_png_images_and_id = defaultdict(list)
    if blobs_and_id:
        for blob_id, blob in blobs_and_id.items():
            logger.info(f"Converting BLOB {blob_id} to images...")
            imagens = converter_blob_para_imagens(blob, "pdf")  # Assuming PDF for this example

            for img in imagens:
                if img_enhancement:
                    img = aplicar_clahe(img)
                img_bytes = imagens_para_bytes(img)
                byte_png_images_and_id[blob_id].append(img_bytes[0])
            logger.info(f"✅ Converted image from BLOB {blob_id} to png bytes.")
    return byte_png_images_and_id

In [ ]:
def extract_text_single_id(images_bytes_list: List[bytes] , textract_instance: TextractKVExtractor, extract_full_text: bool = False) -> str:
    """
    Extract text from a list of PNG image bytes using TextractKVExtractor.
    if extract full text is true, it will not only return key values pairs.
    Args:
        id_and_images: Tuple containing (id, list of PNG image bytes)
        textract_instance: TextractKVExtractor instance

    Returns:
        concatenated extracted text from all images
    """

    full_text = ""
    for img_bytes in images_bytes_list:
        text = textract_instance.run(file_bytes=img_bytes, extract_full_text=extract_full_text)
        full_text += text
    return full_text

In [ ]:
class AnthropicLLMService():
    def __init__(self, model_id: str, model_version: str, client: boto3.client, system_prompt: str, max_tokens: int, temperature: float, budget_tokens: int) -> None:
        self.model_id = model_id
        self.model_version = model_version
        self.client = client
        self.system_prompt = system_prompt
        self.MAX_TOKENS = max_tokens
        self.TEMPERATURE = temperature
        self.BUDGET_TOKENS = budget_tokens

    def _config_body(self, input_str: str) -> Dict[str, Any]:
        body = {
                "anthropic_version": self.model_version,
                "max_tokens": self.MAX_TOKENS,
                "temperature": self.TEMPERATURE,
                "thinking": {
                    "type": "enabled",
                    "budget_tokens": self.BUDGET_TOKENS
                },
                "system": self.system_prompt,
                "messages": [
                    {"role": "user", "content": [{"type": "text", "text": input_str}]},
                ]
            }
        return body

    def _safe_extract_json(self, raw_text: str) -> dict:
        match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', raw_text, re.DOTALL) \
            or re.search(r'\{.*?\}', raw_text, re.DOTALL)
        
        if not match:
            raise ValueError("Nenhum JSON encontrado na resposta do modelo.")

        json_str = match.group(1) if match.lastindex else match.group(0)
        json_str = json_str.strip()

        try:
            return json.loads(json_str)
        except json.JSONDecodeError as e:
            logger.warning(f"JSON inválido detectado: {e}")
            # tenta corrigir
            cleaned = re.sub(r'^[^\{]*', '', json_str)
            cleaned = re.sub(r'[^\}]*$', '', cleaned)
            return json.loads(cleaned)

    def invoke_model(self, input_str: str) -> dict:
        response = self.client.invoke_model(
            body=json.dumps(self._config_body(input_str)),
            modelId=self.model_id
        )
        body_str = response["body"].read().decode("utf-8")  # decode if it's bytes
        if not body_str.strip():
            raise ValueError("Empty body from Bedrock model")

        response_body = json.loads(body_str)

        # Extract text content
        text_parts = [
            block.get("text", "")
            for block in response_body.get("content", [])
            if block.get("type") == "text"
        ]
        raw_text_response = "\n".join(text_parts)
        parsed_json = self._safe_extract_json(raw_text_response)

        return parsed_json



# apos salvar em .py, para usar basta configurar como as celulas abaixo

In [ ]:

## initialize services and variables
app_constants = AppConstants()
settings = Settings()


texttract_client = criar_boto3_client("textract", settings)
bedrock_client = criar_boto3_client("bedrock-runtime", settings)

texttract_instance = TextractKVExtractor(texttract_client)
llm_instance = AnthropicLLMService(
    model_id=settings.BEDROCK_MODEL_ID,
    model_version=settings.BEDROCK_MODEL_VERSION,
    client=bedrock_client,
    system_prompt=system_prompt,
    max_tokens=app_constants.MAX_TOKENS,
    temperature=app_constants.TEMPERATURE,
    budget_tokens=app_constants.BUDGET_TOKENS
)




{"timestamp": "2025-08-14T17:12:31", "level": "INFO", "name": "__main__", "message": "Criando cliente TEXTRACT para a região: us-east-1...", "filename": "788707453.py", "lineno": 5}
{"timestamp": "2025-08-14T17:12:31", "level": "INFO", "name": "__main__", "message": "Cliente TEXTRACT criado com sucesso.", "filename": "788707453.py", "lineno": 15}
{"timestamp": "2025-08-14T17:12:31", "level": "INFO", "name": "__main__", "message": "Criando cliente BEDROCK-RUNTIME para a região: us-east-1...", "filename": "788707453.py", "lineno": 5}


{"timestamp": "2025-08-14T17:12:31", "level": "INFO", "name": "__main__", "message": "Cliente BEDROCK-RUNTIME criado com sucesso.", "filename": "788707453.py", "lineno": 15}


### database processing mocking and running trough

In [ ]:

# mocking images and id_aviso list ############
doc_path = "/home/joao/projects/company_projects/carteirinha-api/documents/pdf_carteirinha/LO_DOCUMENTO_ANEXO_CIRURGICO.pdf"
# getting the pdf blob
with open(doc_path, "rb") as f:
    blob = f.read()
###############################################
id = "123456"
id2 = "394875"
ids_and_blobs = {id: blob,
                 id2: blob}
### get blobs and ids as a dict of str id and bytes pdf from the database processing
byte_png_images_and_ids = process_blobs(ids_and_blobs, img_enhancement=False)

results = {}
for id in byte_png_images_and_ids:
    single_id_full_text = extract_text_single_id(images_bytes_list=byte_png_images_and_ids[id],
                                                 textract_instance=texttract_instance, extract_full_text=False)
    llm_response = llm_instance.invoke_model(input_str=single_id_full_text)
    results[id] = llm_response

    

{"timestamp": "2025-08-14T17:12:31", "level": "INFO", "name": "__main__", "message": "Converting BLOB 123456 to images...", "filename": "3031513262.py", "lineno": 14}
{"timestamp": "2025-08-14T17:12:31", "level": "INFO", "name": "__main__", "message": "Processando PDF com 1 página(s)...", "filename": "3610234635.py", "lineno": 9}
{"timestamp": "2025-08-14T17:12:32", "level": "INFO", "name": "__main__", "message": "✅ Converted image from BLOB 123456 to png bytes.", "filename": "3031513262.py", "lineno": 22}
{"timestamp": "2025-08-14T17:12:32", "level": "INFO", "name": "__main__", "message": "Converting BLOB 394875 to images...", "filename": "3031513262.py", "lineno": 14}
{"timestamp": "2025-08-14T17:12:32", "level": "INFO", "name": "__main__", "message": "Processando PDF com 1 página(s)...", "filename": "3610234635.py", "lineno": 9}
{"timestamp": "2025-08-14T17:12:32", "level": "INFO", "name": "__main__", "message": "✅ Converted image from BLOB 394875 to png bytes.", "filename": "303151

In [ ]:
results

{'123456': {'convenio': 'SUL AMERICA',
  'plano': 'ESPECIAL 100',
  'nome_pessoa': 'MARIA DE LOURDES SILVA CAMARA',
  'numero_carteirinha': '88884837395600020'},
 '394875': {'convenio': 'SUL AMERICA',
  'plano': 'ESPECIAL 100',
  'nome_pessoa': 'MARIA DE LOURDES SILVA CAMARA',
  'numero_carteirinha': '88888483739560020'}}

Bonitao demais. salvar essas funcoes como utils e modulos e reusar  outro notebook

In [ ]:
single_id_full_text

"defaultdict(<class 'list'>, {'Carencias:': ['(1)26/01/2024 (2)09/07/2024 (4)09/07/2024 (5)09/07/2024'], 'SAC:': ['0800 722 0504'], 'Produto': ['557'], 'Titular :': ['MARIA DE LOURDES SILVA CAMARA'], 'Cobertura': ['AMBULATORIAL + HOSPITALAR + OBSTETRICIA'], 'Empresa': ['8WZOD - BIG BOX'], 'Código de Identificação': ['88888 4837 3956 0020'], 'Nascimento': ['03/03/1940'], 'ANS - n°': ['006246'], 'Plano': ['ESPECIAL 100'], 'DEMAIS REGIOES:': ['0800 970 0500'], 'Acomodação': ['APARTAMENTO'], 'CAPITAIS E REG.': [''], 'CNS': ['']})"

# Saved code as mudular utils to api, usage on api style  and others 

## USAGE 1: simple

In [7]:
# trying to use the mudular code to get the workflow feeling
import sys
import os

# add app to path so we dont get the error ModuleNotFoundError: No module named 'app'
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from app.utils.process_images import process_blobs
from app.utils.textract_service import TextractKVExtractor, extract_text_single_id
from app.utils.llm_service import AnthropicLLMService
from app.utils.aws_services_handler import create_boto3_client
from app.utils.config import load_config, AppConstants
from app.utils.logger import get_logger
from app.utils.system_prompts.prompt_handler import Prompts


logger = get_logger(name=__name__)
config_vars = load_config()

In [8]:
app_constants = AppConstants()
texttract_client = create_boto3_client("textract", config_vars)
bedrock_client = create_boto3_client("bedrock-runtime", config_vars)
carteirinha_prompt = Prompts.carteirinha_extraction_prompt

texttract_instance = TextractKVExtractor(texttract_client)
llm_instance = AnthropicLLMService(
    model_id=app_constants.BEDROCK_DEFAULT_MODEL_ID,
    model_version=app_constants.BEDROCK_DEFAULT_MODEL_VERSION,
    client=bedrock_client,
    system_prompt=carteirinha_prompt,
    max_tokens=app_constants.MAX_TOKENS,
    temperature=app_constants.TEMPERATURE,
    budget_tokens=app_constants.BUDGET_TOKENS
)


{"timestamp": "2025-08-14T19:47:54", "level": "INFO", "name": "app.utils.aws_services_handler", "message": "Criando cliente TEXTRACT para a região: us-east-1...", "filename": "aws_services_handler.py", "lineno": 17}
{"timestamp": "2025-08-14T19:47:54", "level": "INFO", "name": "app.utils.aws_services_handler", "message": "Cliente TEXTRACT criado com sucesso.", "filename": "aws_services_handler.py", "lineno": 26}
{"timestamp": "2025-08-14T19:47:54", "level": "INFO", "name": "app.utils.aws_services_handler", "message": "Criando cliente BEDROCK-RUNTIME para a região: us-east-1...", "filename": "aws_services_handler.py", "lineno": 17}
{"timestamp": "2025-08-14T19:47:54", "level": "INFO", "name": "app.utils.aws_services_handler", "message": "Cliente BEDROCK-RUNTIME criado com sucesso.", "filename": "aws_services_handler.py", "lineno": 26}


In [9]:
# usage
# mocking images and id_aviso list ############
doc_path = "/home/joao/projects/company_projects/carteirinha-api/documents/pdf_carteirinha/LO_DOCUMENTO_ANEXO_CIRURGICO.pdf"
# getting the pdf blob
with open(doc_path, "rb") as f:
    blob = f.read()
###############################################
id = "123456"
id2 = "394875"
ids_and_blobs = {id: blob,
                 id2: blob}
### get blobs and ids as a dict of str id and bytes pdf from the database processing
byte_png_images_and_ids = process_blobs(ids_and_blobs, img_enhancement=False, MAX_IMAGES_PER_BLOB=20)

{"timestamp": "2025-08-14T19:47:54", "level": "INFO", "name": "app.utils.process_images", "message": "Converting BLOB 123456 to images...", "filename": "process_images.py", "lineno": 69}
{"timestamp": "2025-08-14T19:47:54", "level": "INFO", "name": "app.utils.process_images", "message": "Processando PDF com 1 página(s)...", "filename": "process_images.py", "lineno": 23}
{"timestamp": "2025-08-14T19:47:55", "level": "INFO", "name": "app.utils.process_images", "message": "✅ Converted image from BLOB 123456 to png bytes.", "filename": "process_images.py", "lineno": 81}
{"timestamp": "2025-08-14T19:47:55", "level": "INFO", "name": "app.utils.process_images", "message": "Converting BLOB 394875 to images...", "filename": "process_images.py", "lineno": 69}
{"timestamp": "2025-08-14T19:47:55", "level": "INFO", "name": "app.utils.process_images", "message": "Processando PDF com 1 página(s)...", "filename": "process_images.py", "lineno": 23}
{"timestamp": "2025-08-14T19:47:55", "level": "INFO", 

In [10]:
results = {}
for id in byte_png_images_and_ids:
    single_id_full_text = extract_text_single_id(images_bytes_list=byte_png_images_and_ids[id],
                                                 textract_instance=texttract_instance, extract_full_text=False)
    llm_response = llm_instance.invoke_model(input_str=single_id_full_text)
    results[id] = llm_response

In [11]:
results

{'123456': {'convenio': 'SUL AMERICA',
  'plano': 'ESPECIAL 100',
  'nome_pessoa': 'MARIA DE LOURDES SILVA CAMARA',
  'numero_carteirinha': '88888483739560020'},
 '394875': {'convenio': None,
  'plano': 'ESPECIAL 100',
  'nome_pessoa': 'MARIA DE LOURDES SILVA CAMARA',
  'numero_carteirinha': '888884837395600200'}}

In [12]:
single_id_full_text

"defaultdict(<class 'list'>, {'Carencias:': ['(1)26/01/2024 (2)09/07/2024 (4)09/07/2024 (5)09/07/2024'], 'SAC:': ['0800 722 0504'], 'Produto': ['557'], 'Titular :': ['MARIA DE LOURDES SILVA CAMARA'], 'Cobertura': ['AMBULATORIAL + HOSPITALAR + OBSTETRICIA'], 'Empresa': ['8WZOD - BIG BOX'], 'Código de Identificação': ['88888 4837 3956 0020'], 'Nascimento': ['03/03/1940'], 'ANS - n°': ['006246'], 'Plano': ['ESPECIAL 100'], 'DEMAIS REGIOES:': ['0800 970 0500'], 'Acomodação': ['APARTAMENTO'], 'CAPITAIS E REG.': [''], 'CNS': ['']})"